In [6]:
import os
import requests
import zipfile
import glob
import pandas as pd
# 1. Définir l’URL du fichier ZIP et les chemins locaux
url = "https://archive.ics.uci.edu/static/public/242/energy+efficiency.zip"
zip_filename = "energy_efficiency.zip"

data_dir = os.path.join("..", "data")

print("Répertoire courant (cwd) :", os.getcwd())
print("Chemin absolu de data_dir  :", os.path.abspath(data_dir))

# 2. Télécharger le ZIP dans le dossier courant du notebook
response = requests.get(url, stream=True)
response.raise_for_status()
with open(zip_filename, "wb") as f:
    for chunk in response.iter_content(chunk_size=8192):
        f.write(chunk)
print(f"Téléchargement terminé : {zip_filename}")

# 3. Créer le répertoire data s’il n’existe pas
os.makedirs(data_dir, exist_ok=True)

# 4. Dézipper le contenu dans ../data
with zipfile.ZipFile(zip_filename, "r") as zip_ref:
    zip_ref.extractall(data_dir)
print(f"Extraction terminée dans : {os.path.abspath(data_dir)}")

xlsx_files = glob.glob(os.path.join(data_dir, "*.xlsx"))
if not xlsx_files:
    raise FileNotFoundError(f"Aucun fichier .xlsx trouvé dans '{data_dir}'")
elif len(xlsx_files) > 1:
    raise RuntimeError(f"Plusieurs fichiers .xlsx trouvés dans '{data_dir}' : {xlsx_files}")
else:
    xlsx_path = xlsx_files[0]

print(f"Fichier Excel trouvé : {xlsx_path}")

# 5. Supprimer le ZIP si vous ne voulez pas le garder
os.remove(zip_filename)

Répertoire courant (cwd) : /workspace/Conso_Energy/notebooks
Chemin absolu de data_dir  : /workspace/Conso_Energy/data
Téléchargement terminé : energy_efficiency.zip
Extraction terminée dans : /workspace/Conso_Energy/data
Fichier Excel trouvé : ../data/ENB2012_data.xlsx


In [7]:
import sys
config_dir = os.path.abspath(os.path.join(os.getcwd(), "..", "config"))
sys.path.append(config_dir)

# Maintenant Python verra C:\Users\nicol\docker_projects\Conso_Energy\config
from column_config import column_mapping

# Affichons le paramétrage :
rows = []
for code, info in column_mapping.items():
    rows.append({
        "Code": code,
        "Nom avec préfixe": info["normalized_name"],
        "Type": info["data_type"]
    })

# Créer et afficher le DataFrame
df = pd.DataFrame(rows, columns=["Code", "Nom avec préfixe", "Type"])
print(df)

  Code             Nom avec préfixe         Type
0   X1       f_relative_compactness      numeric
1   X2               f_surface_area      numeric
2   X3                  f_wall_area      numeric
3   X4                  f_roof_area      numeric
4   X5             f_overall_height      numeric
5   X6                f_orientation  categorical
6   X7               f_glazing_area      numeric
7   X8  f_glazing_area_distribution  categorical
8   Y1               l_heating_load      numeric
9   Y2               l_cooling_load      numeric


In [8]:

# 1. Charger le fichier Excel dézippé à partir de xlsx_path
df = pd.read_excel(xlsx_path)

# 2. Construire et appliquer le dictionnaire de renommage
rename_dict = {orig: info["normalized_name"] for orig, info in column_mapping.items()}
df = df.rename(columns=rename_dict)

# 3. Enregistrer en CSV dans le même répertoire que xlsx_path
data_dir = os.path.dirname(xlsx_path)
output_csv = os.path.join(data_dir+'/csv/', "energy_efficiency.csv")
df.to_csv(output_csv, index=False)

print(f"CSV généré ici : {os.path.abspath(output_csv)}")

CSV généré ici : /workspace/Conso_Energy/data/csv/energy_efficiency.csv
